# Blockchain-IPFS Storage: run the tests and the benchmark

This notebook runs the full workflow on Google Colab: install, contract and crypto tests, deployment to Sepolia, a smoke test, the real benchmark, and the statistics and figures.

**Before you start**
- You need a **new** Pinata key (upload-only scope), a Sepolia RPC URL, and a **dedicated test wallet** funded from a Sepolia faucet. Never use a wallet with real funds.
- Secrets are typed into prompts and written only to a temporary `.env` inside this Colab session. The last cell deletes it. Do not paste secrets into code cells and do not commit `.env`.
- Colab runs in a data centre, so its network differs from a home connection. Record this as your testbed in the paper; results are not comparable to a 50/10 Mbps home link.
- Run the cells in order. Start with `SMOKE = True`, check that everything works, then set `SMOKE = False` and rerun from the configuration cell.

## 1. Node.js 20.6 or newer

In [ ]:
import subprocess, re

def sh(cmd, **kw):
    return subprocess.run(cmd, shell=True, text=True, capture_output=True, **kw)

ver = sh("node --version").stdout.strip()
print("Current Node:", ver or "not found")
major_minor = tuple(int(x) for x in re.findall(r"\d+", ver)[:2]) if ver else (0, 0)
if major_minor < (20, 6):
    print(sh("curl -fsSL https://deb.nodesource.com/setup_20.x | bash - && apt-get install -y nodejs").stdout[-400:])
    print("Now:", sh("node --version").stdout.strip())

## 2. Get the project

Option A (default): upload `blockchain-ipfs-storage.zip` when prompted.
Option B: clone your repository instead (set `REPO_URL`) once it contains these files.

In [ ]:
import os, zipfile

REPO_URL = ""   # e.g. "https://github.com/usra7/blockchain-ipfs-storage.git"; leave empty to upload the zip

os.chdir("/content")
if REPO_URL:
    print(sh(f"rm -rf blockchain-ipfs-storage && git clone {REPO_URL} blockchain-ipfs-storage").stdout)
else:
    from google.colab import files
    uploaded = files.upload()
    name = next(iter(uploaded))
    with zipfile.ZipFile(name) as z:
        z.extractall("/content")
os.chdir("/content/blockchain-ipfs-storage")
print(sorted(os.listdir(".")))

## 2b. Your sample files

Set `USE_MY_SAMPLES = True` to benchmark with your own files `sample_100KB.txt`, `sample_1MB.txt`, `sample_10MB.txt` and `sample_50MB.txt` (the sizes used in the paper). The notebook builds three extra sizes (512 KiB, 5 MiB, 25 MiB) so the size axis has seven points. With `False`, all seven files are built with random content.

In [ ]:
USE_MY_SAMPLES = True

if USE_MY_SAMPLES:
    os.makedirs("data", exist_ok=True)
    print("Select sample_100KB.txt, sample_1MB.txt, sample_10MB.txt, sample_50MB.txt")
    uploaded_samples = files.upload()
    for fname, content in uploaded_samples.items():
        with open(f"data/{fname}", "wb") as f:
            f.write(content)
    for fname in sorted(os.listdir("data")):
        print(fname, os.path.getsize(f"data/{fname}"), "bytes")

## 3. Install dependencies

In [ ]:
print(sh("npm install --no-audit --no-fund").stdout[-1500:])
print(sh("pip install -q numpy scipy pandas matplotlib").stderr[-300:] or "python packages ready")

## 4. Tests (contract and off-chain crypto)

In [ ]:
r = sh("npm test")
print(r.stdout[-3500:], r.stderr[-1500:])

In [ ]:
r = sh("npm run test:crypto")
print(r.stdout[-1800:], r.stderr[-500:])

## 5. Configuration

`SMOKE = True` runs 2 rounds over 3 small files to check the pipeline. Set it to `False` for the real benchmark: 30 rounds over 7 sizes (several hours, and about 210 transactions' worth of Sepolia ETH).

In [ ]:
SMOKE = True
CONFIRMATIONS = 1
GATEWAY = "https://gateway.pinata.cloud/ipfs"

if USE_MY_SAMPLES:
    ALL_FILES = ["data/sample_100KB.txt", "data/sample_512KiB.bin", "data/sample_1MB.txt", "data/sample_5MiB.bin",
                 "data/sample_10MB.txt", "data/sample_25MiB.bin", "data/sample_50MB.txt"]
    EXTRA_SIZES = "512KiB,5MiB,25MiB"   # only these are built; the rest are your files
else:
    ALL_FILES = [f"data/sample_{s}.bin" for s in ["100KiB", "512KiB", "1MiB", "5MiB", "10MiB", "25MiB", "50MiB"]]
    EXTRA_SIZES = ""
RUNS = 2 if SMOKE else 30
FILES = ALL_FILES[:3] if SMOKE else ALL_FILES
OUT = "results/smoke.json" if SMOKE else "results/results.json"
FIGS = "results/figs_smoke" if SMOKE else "results/figs"
print("SMOKE" if SMOKE else "FULL RUN", "| runs:", RUNS, "| files:", len(FILES), "| out:", OUT)

## 6. Secrets (typed at the prompts, kept only in this session)

In [ ]:
import getpass, os

pinata_jwt = getpass.getpass("Pinata JWT (upload-only scope): ")
rpc_url = getpass.getpass("Sepolia RPC URL: ")
private_key = getpass.getpass("Private key of the dedicated TEST wallet (0x...): ")

env_lines = [
    f"PINATA_JWT={pinata_jwt}",
    f"RPC_URL={rpc_url}",
    f"PRIVATE_KEY={private_key}",
    f"FILES={','.join(FILES)}",
    f"RUNS={RUNS}",
    f"CONFIRMATIONS={CONFIRMATIONS}",
    f"GATEWAY={GATEWAY}",
    f"OUT={OUT}",
]
with open(".env", "w") as f:
    f.write("\n".join(env_lines) + "\n")
os.chmod(".env", 0o600)
del pinata_jwt, rpc_url, private_key
print(".env written (permissions 600)")

## 7. Deploy the contract to Sepolia

If you already deployed, put the address in `EXISTING_CONTRACT` to skip deployment. Record the printed address and transaction hash for the paper.

In [ ]:
EXISTING_CONTRACT = ""

if EXISTING_CONTRACT:
    address = EXISTING_CONTRACT
else:
    r = sh("npx hardhat run scripts/deploy.js --network sepolia")
    print(r.stdout, r.stderr[-1500:])
    m = re.search(r"deployed to:\s*(0x[0-9a-fA-F]{40})", r.stdout)
    assert m, "Deployment failed: see the output above"
    address = m.group(1)

with open(".env", "a") as f:
    f.write(f"CONTRACT={address}\n")
print("CONTRACT =", address)

## 8. Build the sample files

In [ ]:
r = sh("npm run samples", env=dict(os.environ, SAMPLE_SIZES=EXTRA_SIZES))
print(r.stdout[-900:], r.stderr[-400:])
missing = [f for f in FILES if not os.path.exists(f)]
assert not missing, f"Missing sample files: {missing}"
print("All benchmark files present:", len(FILES))

## 9. Run the benchmark

Output streams below. Results are saved incrementally to `OUT`, so an interrupted run keeps what finished.

In [ ]:
!node --env-file=.env scripts/measure.js

## 10. Statistics and figures

In [ ]:
r = sh(f"python analysis/stats.py {OUT} {FIGS}")
print(r.stdout[-4500:], r.stderr[-1500:])

In [ ]:
import pandas as pd
from IPython.display import Image, display

summary = pd.read_csv(f"{FIGS}/summary.csv")
display(summary[summary.measure.isin(["total", "confirm", "gasUsed"])].round(3))
for name in ["fig_stages.png", "fig_total.png", "fig_onchain.png"]:
    display(Image(f"{FIGS}/{name}"))

## 11. Record the environment

Save these versions with the results; the paper's testbed section needs them.

In [ ]:
import json, platform, datetime

info = {
    "recordedAt": datetime.datetime.utcnow().isoformat() + "Z",
    "node": sh("node --version").stdout.strip(),
    "npm": sh("npm --version").stdout.strip(),
    "hardhat": sh("npx hardhat --version").stdout.strip(),
    "python": platform.python_version(),
    "platform": platform.platform(),
    "solidity": "0.8.20 (optimizer enabled, runs 200)",
    "contract": address,
    "smoke": SMOKE,
}
os.makedirs("results", exist_ok=True)
with open("results/environment.json", "w") as f:
    json.dump(info, f, indent=2)
print(json.dumps(info, indent=2))

## 12. Download results and remove secrets

Download the results, then delete `.env`. Send me `results/results.json`, `results/figs/summary.csv`, `results/environment.json` and the console output of step 10 to fill the manuscript. After the run, **revoke the Pinata key** and stop using the test wallet.

In [ ]:
import shutil
shutil.make_archive("/content/results_bundle", "zip", "results")
os.remove(".env")
print(".env removed")
from google.colab import files
files.download("/content/results_bundle.zip")